In [3]:
import shutil
import zipfile
from pathlib import Path
import urllib.request

# ----------------------------
# CONFIG
# ----------------------------
URLS = [
    "https://github.com/Raziye-Aghapour/ReEDS_Morris/raw/refs/heads/main/Fuel%20price%20files/alpha_AEO_2025_West_South_Central_morris_000.csv",
    "https://raw.githubusercontent.com/Raziye-Aghapour/ReEDS_Morris/refs/heads/main/Fuel%20price%20files/ng_demand_AEO_2025_West_South_Central_morris_000.csv",
    "https://github.com/Raziye-Aghapour/ReEDS_Morris/raw/refs/heads/main/Fuel%20price%20files/ng_tot_demand_AEO_2025_West_South_Central_morris_000.csv",
]

OUT_DIR = Path("Fuel_price_files_copies")   # folder where CSVs go
ZIP_NAME = "Fuel_price_files_morris_000_to_200.zip"

START_IDX = 0
END_IDX = 200  # inclusive -> 000..200

# ----------------------------
# DOWNLOAD BASE FILES
# ----------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)

base_paths = []
for url in URLS:
    filename = url.split("/")[-1]
    dest = OUT_DIR / filename
    urllib.request.urlretrieve(url, dest)
    base_paths.append(dest)

# ----------------------------
# MAKE COPIES
# ----------------------------
# We'll create versions _000 ... _200 for each base file
# The _000 versions are already downloaded, so we start copying from _001
for i in range(START_IDX + 1, END_IDX + 1):
    suffix = f"{i:03d}"
    for src in base_paths:
        # replace the trailing _000.csv with _{suffix}.csv
        if src.name.endswith("_000.csv"):
            dst_name = src.name.replace("_000.csv", f"_{suffix}.csv")
        else:
            # fallback: append suffix before .csv
            dst_name = src.stem + f"_{suffix}" + src.suffix
        shutil.copyfile(src, OUT_DIR / dst_name)

# Optionally remove the originally downloaded base files if you only want the generated set:
# for src in base_paths:
#     src.unlink(missing_ok=True)

# ----------------------------
# ZIP EVERYTHING
# ----------------------------
zip_path = Path(ZIP_NAME)
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT_DIR.glob("*.csv")):
        # Put them inside a nice folder name inside the zip
        z.write(p, arcname=f"Fuel price files/{p.name}")

print(f"Created: {zip_path.resolve()}")
print(f"Total CSVs in folder: {len(list(OUT_DIR.glob('*.csv')))}")

Created: /content/Fuel_price_files_morris_000_to_200.zip
Total CSVs in folder: 603
